# fNIRS Hemodynamic Pipeline v2 -- Processing & Feature Extraction

**Mục tiêu**: xử lý tín hiệu fNIRS (HbO/HbR/HbT) một cách bài bản, minh bạch,
có thể tái lập, và trích xuất feature theo phase (REST_BEFORE / MEDITATION /
REST_AFTER + EARLY/MIDDLE/LATE) cho bước phân tích thống kê ở notebook riêng
(`fnirs_hemo_eda_stats_v2.ipynb`).

**KHÔNG phải mục tiêu của pipeline này**: kết luận "tăng/giảm hoạt hoá vỏ não"
(cortical activation) -- fNIRS trán đo tín hiệu huyết động hỗn hợp (da đầu +
mạch máu ngoài sọ + có thể có tín hiệu vỏ não), không thể tách bạch bằng
continuous-wave 2 bước sóng không có kênh short-separation. Notebook này CHỈ
tạo ra các đặc trưng huyết động (hemodynamic features) mô tả sự THAY ĐỔI tín
hiệu đo được ở trán giữa các giai đoạn -- diễn giải sinh lý là việc của người
đọc kết quả, có cân nhắc giới hạn ở mục "CHƯA THỂ XÁC NHẬN" cuối notebook.

## So với pipeline v1 (`fnirs_hemo_batch_analysis.ipynb`, KHÔNG bị ghi đè)

| | v1 (cũ) | v2 (notebook này) |
|---|---|---|
| Input | `fnirs_hbo_hbr.csv` (Layer 2, đã tính sẵn HbO/HbR không motion-correction) | Ưu tiên `fnirs_raw.csv` (raw intensity) -> tự tính OD -> TDDR -> MBLL |
| Motion correction | Không có | TDDR (Fishburn et al. 2019) trên Optical Density |
| Filter/detrend | Không có (dùng thẳng dữ liệu Layer 2) | Lowpass 0.2Hz + robust linear detrend, chạy 1 lần trên TOÀN session liên tục trước khi cắt phase |
| Baseline | Không có | Baseline từ REST_BEFORE (median 30-60s cuối phase), áp dụng cho toàn session |
| Transition exclusion | Không có | Cắt bỏ N giây đầu mỗi phase (`full_phase` và `stable_phase` đều được lưu) |
| Event alignment | Không kiểm chứng, clip âm thầm bằng `max()/min()` | Kiểm chứng khớp EEG/fNIRS bằng thời lượng thực đo, ghi rõ mọi lần clip |
| Feature set | mean/SD/slope (`np.polyfit`)/peak-to-peak/max/min | + median/trimmed-mean/MAD/IQR/AUC/Theil-Sen slope/finite-fraction/valid-duration, cả absolute và delta-so-baseline |
| Spectral | Tỉ lệ công suất 0.02-0.1Hz / 0.02-0.5Hz tính trên tín hiệu ĐÃ lowpass 0.2Hz (không nhất quán -- băng 0.1-0.5Hz gần như = 0 sau lowpass) | Cửa sổ trượt 120s/30s, band nhất quán với nhánh tín hiệu dùng để tính |
| QC | 2 flag (`OK`/`SHORT_SEGMENT`/`NO_VALID_DATA`) | Đa tầng: input/signal/event/baseline/statistical, không bao giờ tự xoá dòng |
| Output | 4 file | 7 CSV/JSON + 1 PNG, hậu tố `_v2`, kèm metadata tái lập |

Xem thêm giải thích quyết định thiết kế ở các cell markdown bên dưới, và
danh sách rủi ro/giới hạn ở cuối notebook.

In [ ]:
from pathlib import Path
import hashlib
import importlib
import json
import re
import sys
import traceback
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from tqdm.notebook import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
mne.set_log_level("ERROR")
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Notebook nằm ở notebooks/ -- gốc project cách 2 cấp thư mục.
REPO_ROOT = Path("..").resolve()
LIB_DIR = REPO_ROOT / "lib"
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

import fnirs_pipeline_v2 as fp
importlib.reload(fp)
print("fnirs_pipeline_v2:", fp.__file__, "| PIPELINE_VERSION:", fp.PIPELINE_VERSION)

## Configuration cell

Toàn bộ tham số điều khiển pipeline nằm ở đây -- KHÔNG hardcode rải rác trong
các cell xử lý bên dưới. Mỗi tham số có chú thích lý do chọn giá trị đó.

In [ ]:
PROCESSING_CONFIG = {
    # --- đường dẫn ---
    "layer2_dir": REPO_ROOT / "data" / "analytics" / "layer2",
    "layer1_sessions_path": REPO_ROOT / "data" / "analytics" / "layer1" / "sessions.csv",
    "out_dir": REPO_ROOT / "data" / "analytics" / "fnirs_hemo_v2",

    # --- sample rate ---
    # Không giả định cứng -- load_session_data() ĐO thực tế từ time_seconds
    # và gắn INVALID_SAMPLING_RATE nếu lệch quá 1Hz so với giá trị mong đợi
    # dưới đây (giá trị mong đợi dùng làm fallback khi không đo được).
    "optical_sampling_rate_hz": 100.0,
    "eeg_sampling_rate_hz": 244.0,  # latency trong event_labels_block.csv tính theo sample này (xem lib/event_labels.py)

    # --- MBLL (Modified Beer-Lambert Law) ---
    "source_detector_distance_cm": 3.0,  # xem lib/fnirs_hbo_hbr.py -- CHƯA CÓ xác nhận geometry thực tế từ nhà sản xuất, dùng giá trị mặc định trước đây
    "age_years": 28,  # CHƯA CÓ tuổi thật của từng participant trong metadata hiện có -- dùng hằng số mặc định cho DPF (Scholkmann & Wolf 2013), xem mục "CHƯA THỂ XÁC NHẬN" cuối notebook

    # --- motion correction ---
    "motion_correction_method": "TDDR",  # Fishburn et al. 2019 -- xem lib/fnirs_pipeline_v2.py::tddr(). Không xếp chồng thêm phương pháp aggressive khác (spline/wavelet) theo yêu cầu.
    "motion_correction_on_concentration_fallback": True,  # nếu phải dùng "Trường hợp B" (không có fnirs_raw.csv), vẫn áp TDDR trên chuỗi HbO/HbR có sẵn -- kém lý tưởng hơn TDDR trên OD nhưng vẫn tốt hơn không sửa gì

    # --- filter + detrend (chạy 1 LẦN trên toàn session liên tục, KHÔNG theo từng phase) ---
    "lowpass_hz": 0.2,  # Pinti et al. 2019 -- loại nhiễu tim mạch (~1-2Hz) và hô hấp (~0.2-0.3Hz) khỏi dải huyết động chậm
    "highpass_hz": None,  # tắt mặc định -- robust linear detrend đã xử lý trôi baseline chậm; bật thêm highpass có thể làm méo tín hiệu MEDITATION dài (10 phút, gần biên tần số thấp)
    "detrend_method": "robust_linear_whole_session",  # Theil-Sen (capped-downsample, xem fp._theilslopes_capped) trên TOÀN session, không reset theo phase
    # Option khác đã có (KHÔNG phải mặc định): "robust_median_rolling_90s" -- rolling-median
    # baseline (fp._robust_rolling_median_detrend, cửa sổ config qua detrend_rolling_window_s,
    # mặc định 90s) thay vì 1 độ dốc tuyến tính toàn phiên -- khử được drift PHI TUYẾN mà
    # detrend tuyến tính bỏ sót. Xác nhận qua unit test synthetic (lib/test_fnirs_pipeline_v2.py)
    # là hoạt động đúng thiết kế. Chưa đủ bằng chứng trên dữ liệu thật để
    # đổi mặc định, giữ nguyên "robust_linear_whole_session" cho tới khi có thêm xác nhận.
    # "detrend_rolling_window_s": 90.0,  # chỉ dùng khi detrend_method="robust_median_rolling_90s"

    # --- signal QC (input + signal, trên tín hiệu THÔ trước xử lý) ---
    "min_finite_fraction": 0.95,  # < 95% giá trị hữu hạn -> HIGH_NAN_RATIO

    # --- event alignment ---
    "alignment_offset_tolerance_s": 1.0,  # lệch thời lượng EEG vs fNIRS quá ngưỡng này -> ALIGNMENT_UNCERTAIN (xem fp.validate_and_align_events)
    "min_phase_duration_s": 30.0,  # phase ngắn hơn -> DURATION_MISMATCH

    # --- baseline correction ---
    "baseline_source": "REST_BEFORE",
    "baseline_window_s": 30.0,  # giây CUỐI của REST_BEFORE -- tránh onset transient đầu rest, gần thời điểm chuyển pha nhất nên đại diện tốt nhất cho baseline "ổn định ngay trước can thiệp"
    "baseline_statistic": "median",  # robust hơn mean trước outlier còn sót

    # --- transition exclusion (giây ĐẦU mỗi phase bị loại khỏi stable_phase) ---
    "transition_exclusion_s": {
        "REST_BEFORE": 5.0,
        "MEDITATION": 15.0,   # chuyển từ rest sang thiền -- kỳ vọng transient huyết động rõ hơn do thay đổi hô hấp/tư thế theo hướng dẫn audio
        "REST_AFTER": 5.0,
    },

    # --- meditation sub-periods ---
    "meditation_subperiod_min_duration_s": 30.0,

    # --- spectral (cửa sổ trượt, nhất quán band với nhánh tín hiệu dùng) ---
    "spectral": {
        "window_s": 120.0,
        "step_s": 30.0,
        "low_band": (0.02, 0.10),
        "total_band": (0.02, 0.20),  # === lowpass_hz phía trên -- KHÔNG mở rộng quá cutoff đã lọc (sửa lỗi không nhất quán của v1)
    },

    # --- statistical QC (robust z, flag-only) ---
    "robust_z_threshold": 6.0,

    # --- phase pairs cho bước thay đổi (dùng ở notebook EDA) ---
    "phase_pairs": [
        ("REST_BEFORE", "MEDITATION"),
        ("MEDITATION", "REST_AFTER"),
        ("REST_BEFORE", "REST_AFTER"),
    ],

    "pipeline_version": fp.PIPELINE_VERSION,
}

PROCESSING_CONFIG["out_dir"].mkdir(parents=True, exist_ok=True)
for k, v in PROCESSING_CONFIG.items():
    print(f"{k}: {v}")

## Session discovery

Cùng quy ước với v1: quét `LAYER2_DIR` tìm tất cả session (mọi session có
`fnirs_raw.csv` HOẶC `fnirs_hbo_hbr.csv`, KHÔNG lọc trước theo `session_role`
-- việc lọc/phân nhóm để so sánh thuộc về bước thống kê ở notebook EDA, xử lý
ở đây phải bao trùm hết để không âm thầm bỏ sót session).

In [ ]:
PARTICIPANT_RE = re.compile(r"participant=([^/\\]+)")
STUDY_DAY_RE = re.compile(r"study_day=([^/\\]+)")
SESSION_RE = re.compile(r"session_id=([^/\\]+)")


def parse_identity(path: Path) -> dict:
    s = str(path)
    participant = PARTICIPANT_RE.search(s)
    study_day = STUDY_DAY_RE.search(s)
    session_id = SESSION_RE.search(s)
    return {
        "participant_key": participant.group(1) if participant else "unknown",
        "study_day": int(study_day.group(1)) if study_day and study_day.group(1).isdigit() else None,
        "session_id": session_id.group(1) if session_id else path.parent.name,
    }


def file_sha256(path: Path, chunk_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


LAYER2_DIR = PROCESSING_CONFIG["layer2_dir"]
session_dirs = sorted({p.parent for p in LAYER2_DIR.rglob("fnirs_hbo_hbr.csv")} | {p.parent for p in LAYER2_DIR.rglob("fnirs_raw.csv")})
print(f"Tìm thấy {len(session_dirs)} thư mục session dưới {LAYER2_DIR}")

sessions_l1 = pd.read_csv(PROCESSING_CONFIG["layer1_sessions_path"], low_memory=False) if PROCESSING_CONFIG["layer1_sessions_path"].exists() else pd.DataFrame()
role_lookup = (
    sessions_l1.set_index("session_id")["session_role"].to_dict()
    if not sessions_l1.empty and "session_role" in sessions_l1.columns
    else {}
)

session_index = []
for d in session_dirs:
    identity = parse_identity(d)
    fnirs_raw_p = d / "fnirs_raw.csv"
    hbo_fallback_p = d / "fnirs_hbo_hbr.csv"
    event_p = d / "event_labels_block.csv"
    eeg_p = d / "eeg_raw.edf"
    session_index.append({
        **identity,
        "session_dir": d,
        "fnirs_raw_path": fnirs_raw_p if fnirs_raw_p.exists() else None,
        "hbo_hbr_fallback_path": hbo_fallback_p if hbo_fallback_p.exists() else None,
        "event_csv_path": event_p if event_p.exists() else None,
        "eeg_raw_path": eeg_p if eeg_p.exists() else None,
        "session_role": role_lookup.get(identity["session_id"]),
    })

session_index_df = pd.DataFrame(session_index)
print(f"Có fnirs_raw.csv (Trường hợp A): {session_index_df['fnirs_raw_path'].notna().sum()}/{len(session_index_df)}")
print(f"Có event_labels_block.csv: {session_index_df['event_csv_path'].notna().sum()}/{len(session_index_df)}")
print(f"Có eeg_raw.edf (kiểm chứng alignment): {session_index_df['eeg_raw_path'].notna().sum()}/{len(session_index_df)}")
session_index_df[["participant_key", "study_day", "session_id", "session_role"]].head(10)

## Hàm xử lý 1 session

Gọi đúng thứ tự các hàm trong `lib/fnirs_pipeline_v2.py` theo pipeline đã
thiết kế: load -> validate input -> preprocess TOÀN session liên tục ->
validate/align event -> baseline (1 lần) -> segment phase (full + stable) ->
(nếu MEDITATION) chia EARLY/MIDDLE/LATE -> compute_phase_metrics cho mỗi
segment -> gắn qc_flags -> trả về (phase_rows, event_rows, run_log_entry).

Không có `except: pass` nào -- mọi lỗi được bắt ở vòng lặp ngoài (cell sau)
và ghi đầy đủ `error_type`/`error_message`/`processing_stage` vào run_log.

In [ ]:
def get_eeg_duration_seconds(eeg_path: Path) -> float:
    raw = mne.io.read_raw_edf(str(eeg_path), preload=False, verbose="ERROR")
    return float(raw.n_times / raw.info["sfreq"])


def process_one_session(row: pd.Series, config: dict) -> dict:
    '''Trả về dict {phase_rows, event_rows, wide_row, input_hashes, status}.
    Raise exception tự nhiên nếu lỗi thật sự (không nuốt lỗi) -- caller ở
    vòng lặp ngoài chịu trách nhiệm bắt + ghi run_log.'''
    identity = {
        "participant_key": row["participant_key"],
        "study_day": row["study_day"],
        "session_id": row["session_id"],
        "session_role": row["session_role"],
    }

    input_hashes = {}
    for key in ["fnirs_raw_path", "hbo_hbr_fallback_path", "event_csv_path"]:
        p = row[key]
        if p is not None and Path(p).exists():
            input_hashes[key] = file_sha256(Path(p))

    loaded = fp.load_session_data(
        row["fnirs_raw_path"], row["hbo_hbr_fallback_path"], row["event_csv_path"],
        optical_fs_expected=config["optical_sampling_rate_hz"],
    )

    if loaded["input_signal_stage"] is None:
        return {
            "phase_rows": [], "event_rows": [], "wide_row": None, "input_hashes": input_hashes,
            "status": "NO_INPUT_DATA", "input_qc_flags": loaded["input_qc_flags"],
        }

    signal_qc_flags = []
    for arr, label in [(loaded.get("fnirs_1_raw"), "ch1"), (loaded.get("fnirs_2_raw"), "ch2"),
                        (loaded.get("hbo_uM"), "hbo_raw"), (loaded.get("hbr_uM"), "hbr_raw")]:
        if arr is not None:
            flags, _ = fp.validate_signal(arr, loaded["sampling_rate_hz"], config["min_finite_fraction"], label)
            signal_qc_flags.extend(flags)

    proc = fp.preprocess_continuous_session(loaded, config)

    eeg_duration_s = None
    if row["eeg_raw_path"] is not None:
        eeg_duration_s = get_eeg_duration_seconds(row["eeg_raw_path"])

    events_aligned, event_flags = fp.validate_and_align_events(
        loaded["events"], proc["n_samples_after"], proc["sampling_rate_hz"],
        config["eeg_sampling_rate_hz"], session_duration_from_eeg_s=eeg_duration_s,
    )
    for col, val in identity.items():
        events_aligned[col] = val

    baseline, baseline_flags = fp.compute_baseline(
        proc["hbo_uM"], proc["hbr_uM"], proc["hbt_uM"], proc["sampling_rate_hz"],
        events_aligned, config["baseline_source"], config["baseline_window_s"], config["baseline_statistic"],
    )

    segments = fp.segment_phases(
        events_aligned, proc["n_samples_after"], proc["sampling_rate_hz"],
        config["transition_exclusion_s"], config["min_phase_duration_s"],
    )

    session_level_flags = ";".join(sorted(set(loaded["input_qc_flags"] + signal_qc_flags + event_flags + baseline_flags)))

    if not segments:
        # Không có event_labels_block.csv hợp lệ -- KHÔNG bỏ qua session này
        # (yêu cầu bắt buộc: không âm thầm loại bỏ dữ liệu). Tạo 1 phase
        # "OVERALL" phủ toàn bộ tín hiệu đã xử lý để session vẫn xuất hiện
        # trong output, kèm cờ MISSING_PHASE đã có trong session_level_flags.
        segments = [fp.PhaseSegment(
            description="OVERALL", full_slice=(0, proc["n_samples_after"]), stable_slice=(0, proc["n_samples_after"]),
            full_duration_s=proc["n_samples_after"] / proc["sampling_rate_hz"],
            stable_duration_s=proc["n_samples_after"] / proc["sampling_rate_hz"],
            transition_excluded_s=0.0, qc_flags=["MISSING_PHASE"],
        )]

    phase_rows = []
    all_segments = list(segments)
    for seg in segments:
        all_segments.extend(fp.split_meditation_subperiods(seg, proc["sampling_rate_hz"], min_duration_s=config["meditation_subperiod_min_duration_s"]))

    for seg in all_segments:
        rows = fp.compute_phase_metrics(proc["hbo_uM"], proc["hbr_uM"], proc["hbt_uM"], proc["sampling_rate_hz"], baseline, seg, config["spectral"])
        for r in rows:
            r.update(identity)
            r["session_qc_flags"] = session_level_flags
            r["input_signal_stage"] = proc["input_signal_stage"]
            r["motion_correction_method"] = proc["motion_correction_method"]
            r["motion_fraction_ch1"] = proc["motion_fraction_ch1"]
            r["motion_fraction_ch2"] = proc["motion_fraction_ch2"]
            r["eeg_fnirs_alignment_offset_s"] = events_aligned["alignment_offset_s"].iloc[0] if len(events_aligned) else np.nan
            r.update({f"baseline_{k}" if not k.startswith("baseline_") else k: v for k, v in baseline.items()})
            phase_rows.append(r)

    wide_row = {
        **identity,
        "input_signal_stage": proc["input_signal_stage"],
        "n_samples_before": proc["n_samples_before"],
        "n_samples_after": proc["n_samples_after"],
        "length_unchanged": proc["length_unchanged"],
        "motion_fraction_ch1": proc["motion_fraction_ch1"],
        "motion_fraction_ch2": proc["motion_fraction_ch2"],
        "eeg_fnirs_alignment_offset_s": events_aligned["alignment_offset_s"].iloc[0] if len(events_aligned) else np.nan,
        "session_qc_flags": session_level_flags,
    }
    for variant in ["full_phase", "stable_phase"]:
        for r in [rr for rr in phase_rows if rr["feature_variant"] == variant]:
            phase = r["phase"]
            for k, v in r.items():
                if k.startswith(("hbo_", "hbr_", "hbt_")):
                    wide_row[f"{phase}__{variant}__{k}"] = v

    return {
        "phase_rows": phase_rows, "event_rows": events_aligned.to_dict("records"),
        "wide_row": wide_row, "input_hashes": input_hashes, "status": "OK",
        "input_qc_flags": loaded["input_qc_flags"],
    }

## Vòng lặp chính -- chạy toàn bộ session

Mọi lỗi được bắt và ghi lại đầy đủ ngữ cảnh (loại lỗi, thông điệp, session
nào, participant nào) vào `run_log` -- KHÔNG bao giờ bắt lỗi rồi bỏ qua âm
thầm (`except: pass`).

In [ ]:
all_phase_rows = []
all_event_rows = []
all_wide_rows = []
run_log = []
all_input_hashes = {}

for _, row in tqdm(session_index_df.iterrows(), total=len(session_index_df), desc="fNIRS hemo v2"):
    session_id = row["session_id"]
    try:
        result = process_one_session(row, PROCESSING_CONFIG)
        all_input_hashes[session_id] = result["input_hashes"]
        if result["status"] != "OK":
            run_log.append({
                "session_id": session_id, "participant_key": row["participant_key"], "study_day": row["study_day"],
                "status": result["status"], "error_type": None, "error_message": None,
                "processing_stage": "load_session_data",
                "input_qc_flags": ";".join(result.get("input_qc_flags", [])),
                "n_phase_rows": 0,
            })
            continue
        all_phase_rows.extend(result["phase_rows"])
        all_event_rows.extend(result["event_rows"])
        all_wide_rows.append(result["wide_row"])
        run_log.append({
            "session_id": session_id, "participant_key": row["participant_key"], "study_day": row["study_day"],
            "status": "OK", "error_type": None, "error_message": None, "processing_stage": None,
            "input_qc_flags": ";".join(result.get("input_qc_flags", [])),
            "n_phase_rows": len(result["phase_rows"]),
        })
    except Exception as exc:
        run_log.append({
            "session_id": session_id, "participant_key": row["participant_key"], "study_day": row["study_day"],
            "status": "ERROR", "error_type": type(exc).__name__, "error_message": str(exc),
            "processing_stage": "process_one_session",
            "input_qc_flags": None, "n_phase_rows": 0,
        })
        print(f"[ERROR] {row['participant_key']} day={row['study_day']} {session_id}: {type(exc).__name__}: {exc}")
        traceback.print_exc(limit=3)

run_log_df = pd.DataFrame(run_log)
print("\nRun status:")
print(run_log_df["status"].value_counts().to_string())

In [ ]:
phase_df = pd.DataFrame(all_phase_rows)
events_df = pd.DataFrame(all_event_rows)
wide_df = pd.DataFrame(all_wide_rows)
print("phase_df:", phase_df.shape, "| events_df:", events_df.shape, "| wide_df:", wide_df.shape)
phase_df.head(3)

## Gắn cờ QC tổng hợp + phát hiện outlier thống kê

`assign_qc_flags` chỉ phân loại cờ đã có theo tầng (không tạo cờ mới).
`robust_outlier_flags` chạy RIÊNG theo `(phase, feature_variant)` để so sánh
đúng session với session cùng loại phase/biến thể.

In [ ]:
phase_df = fp.assign_qc_flags(phase_df)

metric_cols_for_outlier = [c for c in phase_df.columns if re.match(r"^(hbo|hbr|hbt)_(absolute|delta)_(mean|median)_uM$", c)]
phase_df = fp.robust_outlier_flags(phase_df, metric_cols_for_outlier, ["phase", "feature_variant"], PROCESSING_CONFIG["robust_z_threshold"])

print("qc_pass:", phase_df["qc_pass"].value_counts().to_dict())
print("robust_outlier:", phase_df["robust_outlier"].value_counts().to_dict())
phase_df.groupby(["phase", "feature_variant"])["qc_pass"].mean().rename("fraction_qc_pass")

## Session-level changes (paired phase-to-phase)

Input cho bước thống kê ở notebook EDA -- KHÔNG tính thống kê ở đây, chỉ tạo
bảng "change" per session.

In [ ]:
metric_cols_for_changes = [c for c in phase_df.columns if re.match(r"^(hbo|hbr|hbt)_(absolute|delta)_(mean|median|trimmed_mean10)_uM$", c) or c.endswith("_slope_theilsen_uM_per_min") or c == "hbo_spectral_low_freq_power_ratio"]

changes_df = fp.compute_session_changes(
    phase_df, PROCESSING_CONFIG["phase_pairs"], metric_cols_for_changes,
    group_cols=["session_id", "participant_key", "study_day"],
    feature_variant="stable_phase",
)
print("changes_df:", changes_df.shape)
changes_df.head(10)

## Boxplot tổng quan (file output thứ 8)

So HbO trung bình (`stable_phase`, baseline-relative) giữa các phase chính --
chỉ mục đích QC trực quan nhanh, phân tích thống kê đầy đủ nằm ở notebook
EDA riêng.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
main_phases = ["REST_BEFORE", "MEDITATION", "REST_AFTER"]
plot_df = phase_df[(phase_df["feature_variant"] == "stable_phase") & (phase_df["phase"].isin(main_phases))]
data_by_phase = [plot_df.loc[plot_df["phase"] == ph, "hbo_delta_mean_uM"].dropna().to_numpy() for ph in main_phases]
ax.boxplot(data_by_phase, tick_labels=main_phases, showmeans=True)
ax.set_ylabel("ΔHbO trung bình (µM, so với baseline REST_BEFORE)")
ax.set_title("Phân bố ΔHbO trung bình theo phase (stable_phase, toàn bộ session)")
ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
fig.tight_layout()
boxplot_path = PROCESSING_CONFIG["out_dir"] / "fnirs_hemo_by_phase_boxplot_v2.png"
fig.savefig(boxplot_path, dpi=150)
plt.show()
print("Saved:", boxplot_path)

## QC summary + Export kết quả

`qc_summary_df` đếm số dòng theo từng flag (để biết flag nào phổ biến, phục
vụ audit) -- xem thêm `fnirs_hemo_qc_summary_v2.csv`.

## Gắn thêm cờ chất lượng HAPPE (EEG cùng session, tham khảo)

Join thêm cột chất lượng từ báo cáo QC của HAPPE (`eeg_happe_qc.ipynb`,
chạy trên EEG 2 kênh AF3/AF4 CÙNG session) theo `session_id` -- đây là chỉ
số CẤP THIẾT BỊ/PHIÊN (thời lượng ghi + % phương sai EEG giữ lại sau làm
sạch artefact), KHÔNG PHẢI chỉ số trực tiếp đo chất lượng tín hiệu fNIRS.
Dùng như 1 biến hiệp phương sai/lọc độ nhạy bổ sung (session cùng 1 lần đeo
thiết bị, tiếp xúc da kém có thể ảnh hưởng cả 2 loại cảm biến), KHÔNG dùng
để tự động loại bỏ dữ liệu fNIRS nào -- xem `lib/happe_qc.py` để biết đúng
cách tính ngưỡng.

In [ ]:
import happe_qc
importlib.reload(happe_qc)

HAPPE_QC_DIR = REPO_ROOT / "data" / "analytics" / "happe_qc"
happe_quality_df = happe_qc.load_happe_quality_table(HAPPE_QC_DIR)

if happe_quality_df.empty:
    print("Chưa tìm thấy output của eeg_happe_qc.ipynb -- bỏ qua bước join (không bắt buộc).")
    for col in ["happe_file_length_s", "happe_pct_var_retained_postwavelet", "happe_pct_good_channels", "happe_qc_flags", "happe_qc_pass"]:
        phase_df[col] = pd.NA
else:
    phase_df = phase_df.merge(happe_quality_df, on="session_id", how="left")
    n_matched = phase_df["happe_qc_pass"].notna().sum()
    print(f"Đã join HAPPE QC: {n_matched}/{len(phase_df)} dòng khớp session_id (session chưa chạy HAPPE sẽ có NA, không phải False).")
    print(phase_df["happe_qc_flags"].value_counts(dropna=False).to_string())

In [ ]:
flag_counts = {}
for flags_str in phase_df["qc_flags"].dropna():
    for f in flags_str.split(";"):
        if f:
            flag_counts[f] = flag_counts.get(f, 0) + 1
qc_summary_df = pd.DataFrame(sorted(flag_counts.items(), key=lambda kv: -kv[1]), columns=["qc_flag", "n_rows"])
print(qc_summary_df.to_string(index=False))

# hash phẳng {session_id: {file: hash}} -> {session_id::file: hash} để lưu JSON gọn
flat_hashes = {f"{sid}::{k}": v for sid, d in all_input_hashes.items() for k, v in d.items()}

paths = fp.export_results(
    PROCESSING_CONFIG["out_dir"], phase_df, wide_df, changes_df, events_df,
    run_log_df, qc_summary_df, PROCESSING_CONFIG, flat_hashes,
)
for k, v in paths.items():
    print(k, "->", v)

## Validation checks (mục đích: bắt lỗi ngay trong notebook, không để lộ ra bước thống kê)

Các kiểm tra cơ bản nhất -- danh sách đầy đủ 15 kiểm tra bắt buộc của toàn bộ
pipeline (bao gồm cả notebook EDA) nằm ở tài liệu deliverables riêng.

In [ ]:
assert phase_df["session_id"].notna().all(), "Có dòng thiếu session_id"
assert (phase_df["feature_variant"].isin(["full_phase", "stable_phase"])).all()
assert wide_df.empty or wide_df["length_unchanged"].all(), "Có session bị đổi độ dài tín hiệu ngoài ý muốn khi preprocess"
assert not (phase_df["qc_flags"].isna()).any(), "qc_flags không được để trống (phải là chuỗi rỗng '' nếu không có cờ)"
n_ok = (run_log_df["status"] == "OK").sum()
n_total = len(run_log_df)
print(f"Session OK: {n_ok}/{n_total}")
print(f"Session lỗi (ERROR): {(run_log_df['status'] == 'ERROR').sum()}")
print(f"Session thiếu input: {(run_log_df['status'] == 'NO_INPUT_DATA').sum()}")
assert n_ok > 0, "Không có session nào xử lý thành công"
print("\nTất cả kiểm tra cơ bản PASSED.")